# **Feature Extraction with ResNet50 (PrePATH)**
Extracting features for TCGA LUAD/LUSC patches using the ResNet50 vision encoder.

### **0. Setup and Dependencies**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/birkhoffkiki/PrePATH
%cd PrePATH

In [ ]:
!pip install torch==2.3.0 \
torchvision==0.18.0 \
timm==1.0.15 \
wandb \
openslide-python \
openslide-bin h5py==3.8.0 \
numpy==1.26.4 \
opencv_python==4.7.0.72 \
opencv_python_headless==4.10.0.84 \
paramiko==3.5.1 \
pathos==0.3.3 \
Pillow==11.2.1 \
PyYAML==6.0.2 \
scipy==1.15.3 \
scp==0.15.0 \
setuptools==65.6.3 \
pandas einops_exts

In [ ]:
!pip install --upgrade pip setuptools wheel Cython

%cd /content/

!git clone https://github.com/MrPeterJin/ASlide.git

%cd /content/ASlide
!pip install .

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

### **1. Configuration**
Set the cancer type (`luad` or `lusc`) and verify paths.

In [ ]:
!ls /content/drive/MyDrive/CS231/tcga_dataset/luad_patches/patches | wc

In [ ]:
!ls /content/drive/MyDrive/CS231/tcga_dataset/lusc_patches/patches | wc

In [ ]:
import os

# Spatial coordinates (H5 files) source parent
DATA_COORS_DIR_LUAD = f"/content/drive/MyDrive/CS231/tcga_dataset/luad_patches"
# Exact directory where .h5 files are stored
H5_DIR_LUAD = os.path.join(DATA_COORS_DIR_LUAD, "patches")

# Raw data (WSIs) root directory
WSI_DIR_LUAD = f"/content/drive/MyDrive/CS231/tcga_dataset/luad"

# Destination for features
FEAT_DIR_LUAD = f"/content/drive/MyDrive/CS231/luad_feats"

TASK_NAME_LUAD = f"luad_resnet50"
CSV_PATH_LUAD = f"csv/{TASK_NAME_LUAD}"

os.makedirs(FEAT_DIR_LUAD, exist_ok=True)
os.makedirs(CSV_PATH_LUAD, exist_ok=True)

# Spatial coordinates (H5 files) source parent
DATA_COORS_DIR_LUSC = f"/content/drive/MyDrive/CS231/tcga_dataset/lusc_patches"
# Exact directory where .h5 files are stored
H5_DIR_LUSC = os.path.join(DATA_COORS_DIR_LUSC, "patches")

# Raw data (WSIs) root directory
WSI_DIR_LUSC = f"/content/drive/MyDrive/CS231/tcga_dataset/lusc"

# Destination for features
FEAT_DIR_LUSC = f"/content/drive/MyDrive/CS231/lusc_feats"

TASK_NAME_LUSC = f"lusc_resnet50"
CSV_PATH_LUSC = f"csv/{TASK_NAME_LUSC}"

os.makedirs(FEAT_DIR_LUSC, exist_ok=True)
os.makedirs(CSV_PATH_LUSC, exist_ok=True)

### **2. Generate Slide List**
This scans the coordinates folder to create a processing list.

In [ ]:
%cd /content/PrePATH

!python scripts/extract_feature/generate_csv.py \
        --h5_dir "{H5_DIR_LUSC}" \
        --num 1 \
        --root "{CSV_PATH_LUSC}"

### **3. Extract Features**
Using `--model "resnet50"` (ImageNet pretrained by default in PrePATH).

In [ ]:
!python extract_features_fp_fast.py \
        --model "resnet50" \
        --csv_path "./{CSV_PATH_LUSC}/part_0.csv" \
        --data_coors_dir "{DATA_COORS_DIR_LUSC}" \
        --data_slide_dir "{WSI_DIR_LUSC}" \
        --feat_dir "{FEAT_DIR_LUSC}" \
        --ignore_partial yes \
        --batch_size 128 \
        --datatype auto \
        --slide_ext ".svs" \
        --save_storage "yes"

### **4. Verification**
Check one of the generated `.pt` files.

In [ ]:
import torch
import glob

pt_files = glob.glob(f"{FEAT_DIR_LUSC}/pt_files/resnet50/*.pt")
if pt_files:
    features = torch.load(pt_files[0])
    print(f"Loaded: {pt_files[0]}")
    print(f"Feature Shape: {features.shape}") # Should be [num_patches, 1024] for truncated ResNet50
else:
    print("No features extracted yet.")